# Tạo database dùng Sqlite

In [2]:
import sqlite3
import pandas as pd

In [3]:
conn = sqlite3.connect("../data/database/career.db")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

onet_path = "../data/raw/onet"

In [3]:
# 1. Occupations
cur.execute("""
CREATE TABLE IF NOT EXISTS occupations (
    occupation_code   TEXT PRIMARY KEY,
    occupation_title  TEXT NOT NULL,
    description       TEXT,
    education_level   TEXT,
    job_zone          INTEGER,
    ai_exposure_score REAL
)
""")

# 2. Occupation Skills
cur.execute("""
CREATE TABLE IF NOT EXISTS occupation_skills (
    occupation_code   TEXT NOT NULL,
    skill_id          TEXT NOT NULL,
    skill_name        TEXT,
    importance_score  REAL,
    level_score       REAL,
    normalized_weight REAL,

    PRIMARY KEY (occupation_code, skill_id),

    FOREIGN KEY (occupation_code)
        REFERENCES occupations(occupation_code)
)
""")

# 3. Occupation Tasks
cur.execute("""
CREATE TABLE IF NOT EXISTS occupation_tasks (
    task_id         INTEGER PRIMARY KEY,
    occupation_code TEXT NOT NULL,
    task_statement  TEXT,
    task_type       TEXT,

    FOREIGN KEY (occupation_code)
        REFERENCES occupations(occupation_code)
)
""")

# 4. Engineered Occupation Features
cur.execute("""
CREATE TABLE IF NOT EXISTS occupation_features (
    occupation_code     TEXT PRIMARY KEY,
    cognitive_intensity REAL,
    social_intensity    REAL,
    physical_intensity  REAL,
    digital_intensity   REAL,
    routine_intensity   REAL,

    FOREIGN KEY (occupation_code)
        REFERENCES occupations(occupation_code)
)
""")

# 5. Related Occupations
cur.execute("""
CREATE TABLE IF NOT EXISTS related_occupations (
    occupation_code         TEXT NOT NULL,
    related_occupation_code TEXT NOT NULL,
    relatedness_tier        INTEGER,
    index_order             INTEGER,

    PRIMARY KEY (occupation_code, related_occupation_code),

    FOREIGN KEY (occupation_code)
        REFERENCES occupations(occupation_code),

    FOREIGN KEY (related_occupation_code)
        REFERENCES occupations(occupation_code)
)
""")

conn.commit()
conn.close()

print("Đã tạo career.db")

Đã tạo career.db


# Import data 

In [ ]:

df = pd.read_csv(f"{onet_path}/essential_skills.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head())



(18200, 15)
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']
  O*NET-SOC Code             Title Element ID           Element Name Scale ID  \
0     11-1011.00  Chief Executives    2.A.1.a  Reading Comprehension       IM   
1     11-1011.00  Chief Executives    2.A.1.a  Reading Comprehension       LV   
2     11-1011.00  Chief Executives    2.A.1.b       Active Listening       IM   
3     11-1011.00  Chief Executives    2.A.1.b       Active Listening       LV   
4     11-1011.00  Chief Executives    2.A.1.c                Writing       IM   

   Scale Name  Data Value  N  Standard Error  Lower CI Bound  Upper CI Bound  \
0  Importance        4.12  8          0.1250          3.8800          4.3700   
1       Level        4.62  8          0.1830          4.2664          4.9836   
2  Importance        4.00  8          0.0

In [9]:
occupation_df = pd.read_csv(
    "../data/raw/onet/occupation_data.csv"
)

print("\nShape:")
print(occupation_df.shape)

print("\nColumns:")
print(occupation_df.columns.tolist())

print("\nFirst 5 rows:")
display(occupation_df.head())


Shape:
(1016, 3)

Columns:
['O*NET-SOC Code', 'Title', 'Description']

First 5 rows:


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [7]:
# Import bảng occupations

# =========================
# 1. OCCUPATIONS
# =========================

occupation_df = pd.read_csv(
    f"{onet_path}/occupation_data.csv"
)

# Đổi tên cột
occupation_df = occupation_df.rename(columns={
    "O*NET-SOC Code": "occupation_code",
    "Title": "occupation_title",
    "Description": "description"
})

# Chỉ giữ những cột cần thiết
occupation_df = occupation_df[
    [
        "occupation_code",
        "occupation_title",
        "description"
    ]
]

# Kiểm tra dữ liệu
print("Shape:", occupation_df.shape)
print("Duplicate codes:",
      occupation_df["occupation_code"].duplicated().sum())

print("\nMissing values:")
print(occupation_df.isna().sum())

display(occupation_df.head())

Shape: (1016, 3)
Duplicate codes: 0

Missing values:
occupation_code     0
occupation_title    0
description         0
dtype: int64


,occupation_code,occupation_title,description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [14]:
occupation_df.to_sql(
    "occupations",
    conn,
    if_exists="append",
    index=False
)

print("Imported occupations:", len(occupation_df))

Imported occupations: 1016


In [ ]:
# 2. import bảng OCCUPATION SKILLS
skills_df = pd.read_csv(
    f"{onet_path}/essential_skills.csv"
)

print("Raw skills:", skills_df.shape)

print(skills_df["Scale ID"].value_counts())

Raw skills: (18200, 15)
Scale ID
IM    9100
LV    9100
Name: count, dtype: int64


In [5]:

# Chỉ giữ các cột cần thiết
skills_df = skills_df[
    [
        "O*NET-SOC Code",
        "Element ID",
        "Element Name",
        "Scale ID",
        "Scale Name",
        "Data Value"
    ]
]

# Chuyển:
#
# Importance → cột importance_score
# Level      → cột level_score

skills_df = skills_df.pivot_table(
    index=[
        "O*NET-SOC Code",
        "Element ID",
        "Element Name"
    ],
    columns="Scale ID",
    values="Data Value"
).reset_index()

# Đổi tên
skills_df = skills_df.rename(columns={
    "O*NET-SOC Code": "occupation_code",
    "Element ID": "skill_id",
    "Element Name": "skill_name",
    "IM": "importance_score",
    "LV": "level_score"
})

# Chỉ giữ các cột của database
skills_df = skills_df[
    [
        "occupation_code",
        "skill_id",
        "skill_name",
        "importance_score",
        "level_score"
    ]
]

# normalized_weight chưa tính ở bước raw import
skills_df["normalized_weight"] = None

print("Processed skills:", skills_df.shape)

display(skills_df.head())

Processed skills: (9100, 6)


Scale ID,occupation_code,skill_id,skill_name,importance_score,level_score,normalized_weight
0,11-1011.00,2.A.1.a,Reading Comprehension,4.12,4.62,None
1,11-1011.00,2.A.1.b,Active Listening,4.00,4.75,None
2,11-1011.00,2.A.1.c,Writing,4.12,4.38,None
3,11-1011.00,2.A.1.d,Speaking,4.25,4.75,None
4,11-1011.00,2.A.1.e,Mathematics,3.25,3.50,None


In [6]:
skills_df.to_sql(
    "occupation_skills",
    conn,
    if_exists="append",
    index=False
)

print("Đã import occupation_skills!")

Đã import occupation_skills!


Bổ sung thêm thông tin vào bảng occupation

In [7]:
job_zone_df = pd.read_csv(
    f"{onet_path}/job_zones.csv"
)

print("Shape:", job_zone_df.shape)
print("Columns:", job_zone_df.columns.tolist())

display(job_zone_df.head())

Shape: (923, 5)
Columns: ['O*NET-SOC Code', 'Title', 'Job Zone', 'Date', 'Domain Source']


,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst
2,11-1021.00,General and Operations Managers,4,08/2023,Analyst
3,11-1031.00,Legislators,4,06/2008,Analyst
4,11-2011.00,Advertising and Promotions Managers,4,08/2026,Analyst


In [9]:
job_zone_df = job_zone_df[
    [
        "O*NET-SOC Code",
        "Job Zone"
    ]
].copy()

job_zone_df = job_zone_df.rename(columns={
    "O*NET-SOC Code": "occupation_code",
    "Job Zone": "job_zone"
})

display(job_zone_df.head())

,occupation_code,job_zone
0,11-1011.00,5
1,11-1011.03,5
2,11-1021.00,4
3,11-1031.00,4
4,11-2011.00,4


In [10]:
print("Number of occupations:", job_zone_df["occupation_code"].nunique())

print("\nMissing:")
print(job_zone_df.isna().sum())

print("\nJob Zone distribution:")
print(job_zone_df["job_zone"].value_counts().sort_index())

Number of occupations: 923

Missing:
occupation_code    0
job_zone           0
dtype: int64

Job Zone distribution:
job_zone
2    337
3    208
4    226
5    152
Name: count, dtype: int64


In [11]:
for _, row in job_zone_df.iterrows():

    conn.execute("""
        UPDATE occupations
        SET job_zone = ?
        WHERE occupation_code = ?
    """, (
        row["job_zone"],
        row["occupation_code"]
    ))

conn.commit()

print("Đã cập nhật Job Zone!")

Đã cập nhật Job Zone!


## occupation_task

In [13]:
tasks_raw = pd.read_csv(
    f"{onet_path}/task_statements.csv"
)

print("Shape:", tasks_raw.shape)
print("Columns:")
print(tasks_raw.columns.tolist())

display(tasks_raw.head())

Shape: (18838, 8)
Columns:
['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']


,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,94.0,08/2023,Incumbent
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,Core,95.0,08/2023,Incumbent


In [14]:
tasks = tasks_raw[
    [
        "Task ID",
        "O*NET-SOC Code",
        "Task",
        "Task Type"
    ]
].copy()

tasks = tasks.rename(columns={
    "Task ID": "task_id",
    "O*NET-SOC Code": "occupation_code",
    "Task": "task_statement",
    "Task Type": "task_type"
})

display(tasks.head())

,task_id,occupation_code,task_statement,task_type
0,8823,11-1011.00,Direct or coordinate an organization's financi...,Core
1,8824,11-1011.00,"Confer with board members, organization offici...",Core
2,8827,11-1011.00,"Prepare budgets for approval, including those ...",Core
3,8826,11-1011.00,"Direct, plan, or implement policies, objective...",Core
4,8834,11-1011.00,Prepare or present reports concerning activiti...,Core


In [17]:
print("Total tasks:", len(tasks))
print("Unique task IDs:", tasks["task_id"].nunique())
print("Kiểm tra NaN")
print(tasks.isna().sum())

Total tasks: 18838
Unique task IDs: 18838
Kiểm tra NaN
task_id              0
occupation_code      0
task_statement       0
task_type          418
dtype: int64


In [18]:
missing_row = tasks[tasks["task_type"].isna()]

print(missing_row.head())

    task_id occupation_code  \
66    15267      11-1031.00   
67    15268      11-1031.00   
68    15269      11-1031.00   
69    15270      11-1031.00   
70    15271      11-1031.00   

                                       task_statement task_type  
66  Analyze and understand the local and national ...       NaN  
67  Appoint nominees to leadership posts, or appro...       NaN  
68  Confer with colleagues to formulate positions ...       NaN  
69  Debate the merits of proposals and bill amendm...       NaN  
70  Develop expertise in subject matters related t...       NaN  


In [21]:
# Kiểm tra occupation có tồn tại trong occupations
valid_codes = set(
    occupation_df["occupation_code"]
)

invalid_tasks = tasks[
    ~tasks["occupation_code"].isin(valid_codes)
]

print(
    "Tasks with invalid occupation:",
    len(invalid_tasks)
)

Tasks with invalid occupation: 0


In [22]:
# Insert vào database
tasks.to_sql(
    "occupation_tasks",
    conn,
    if_exists="append",
    index=False
)

print("Đã import occupation_tasks!")

Đã import occupation_tasks!


## Occupation_related

In [4]:
related_raw = pd.read_csv(
    f"{onet_path}/related_occupations.csv"
)

print("Shape:", related_raw.shape)
print("Columns:")
print(related_raw.columns.tolist())

display(related_raw.head())

Shape: (18460, 6)
Columns:
['O*NET-SOC Code', 'Title', 'Related O*NET-SOC Code', 'Related Title', 'Relatedness Tier', 'Index']


,O*NET-SOC Code,Title,Related O*NET-SOC Code,Related Title,Relatedness Tier,Index
0,11-1011.00,Chief Executives,11-1021.00,General and Operations Managers,Primary-Short,1
1,11-1011.00,Chief Executives,13-1111.00,Management Analysts,Primary-Short,2
2,11-1011.00,Chief Executives,11-3031.01,Treasurers and Controllers,Primary-Short,3
3,11-1011.00,Chief Executives,11-9151.00,Social and Community Service Managers,Primary-Short,4
4,11-1011.00,Chief Executives,11-2032.00,Public Relations Managers,Primary-Short,5


In [5]:
related = related_raw[
    [
        "O*NET-SOC Code",
        "Related O*NET-SOC Code",
        "Relatedness Tier",
        "Index"
    ]
].copy()

related = related.rename(columns={
    "O*NET-SOC Code": "occupation_code",
    "Related O*NET-SOC Code": "related_occupation_code",
    "Relatedness Tier": "relatedness_tier",
    "Index": "index_order"
})

display(related.head())

,occupation_code,related_occupation_code,relatedness_tier,index_order
0,11-1011.00,11-1021.00,Primary-Short,1
1,11-1011.00,13-1111.00,Primary-Short,2
2,11-1011.00,11-3031.01,Primary-Short,3
3,11-1011.00,11-9151.00,Primary-Short,4
4,11-1011.00,11-2032.00,Primary-Short,5


In [6]:
# Kiểm tra missing value và duplicate
print("Missing values:")
print(related.isna().sum())

print(
    "Duplicate pairs:",
    related.duplicated(
        subset=[
            "occupation_code",
            "related_occupation_code"
        ]
    ).sum()
)

Missing values:
occupation_code            0
related_occupation_code    0
relatedness_tier           0
index_order                0
dtype: int64
Duplicate pairs: 0


In [8]:
#Kiểm tra occupation có tồn tại
valid_codes = set(
    occupation_df["occupation_code"]
)

invalid_source = related[
    ~related["occupation_code"].isin(valid_codes)
]

invalid_target = related[
    ~related["related_occupation_code"].isin(valid_codes)
]

print(
    "Invalid source occupations:",
    len(invalid_source)
)

print(
    "Invalid related occupations:",
    len(invalid_target)
)

Invalid source occupations: 0
Invalid related occupations: 0


In [9]:
print(
    related["relatedness_tier"].value_counts()
)

relatedness_tier
Supplemental     9230
Primary-Short    4615
Primary-Long     4615
Name: count, dtype: int64


In [10]:
related.to_sql(
    "related_occupations",
    conn,
    if_exists="append",
    index=False
)

print("Đã import related_occupations!")

Đã import related_occupations!


## Import education_level vào bảng occupation

In [11]:
education_raw = pd.read_csv(
    f"{onet_path}/education.csv"
)

print("Shape:", education_raw.shape)

print("\nColumns:")
print(education_raw.columns.tolist())

display(education_raw.head(20))

Shape: (11495, 15)

Columns:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Category', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Date', 'Domain Source']


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Date,Domain Source
0,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),1.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
1,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),2.0,4.46,28,4.1428,0.6307,25.5524,N,08/2023,Incumbent
2,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),3.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
3,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),4.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
4,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),5.0,5.15,28,5.2236,0.6000,32.7756,N,08/2023,Incumbent
5,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),6.0,32.29,28,12.7266,12.6247,61.1586,N,08/2023,Incumbent
6,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),7.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
7,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),8.0,45.91,28,16.6480,17.6593,77.0570,N,08/2023,Incumbent
8,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),9.0,3.94,28,4.0634,0.4511,27.0734,N,08/2023,Incumbent
9,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),10.0,0.55,28,0.5831,0.0637,4.6529,N,08/2023,Incumbent
